# Part 1 — Document Extraction & Prompt Engineering

**Source document:** *Analysis of Revenue and Expenditure, Financial Year 2024* (Singapore Ministry of Finance, distributed Budget Day 16 February 2024) — 37 pages.

**Goal:** extract five structured fields via LLM prompting, with evidence for each.

| # | Field | Type | Stated page |
|---|-------|------|------|
| 1 | Amount of Corporate Income Tax in 2024 | `float` | 5 |
| 2 | YOY percentage difference of Corp Income Tax in 2024 | `float` | 5 |
| 3 | Total amount of top ups in 2024 | `float` | 20 |
| 4 | List of taxes mentioned in section "Operating Revenue" | `list[str]` | 5–6 |
| 5 | Latest Actual Fiscal Position in billions | `float` | 8 |

**Approach:** parse with a deliberately-chosen PDF library → scope the context to only the relevant pages → constrain the LLM with a Pydantic schema that demands a verbatim quote and page number alongside every figure → validate the output against values read by hand from the PDF.

## 0. Setup

```bash
python -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

Then put the free [AI Studio](https://aistudio.google.com/apikey) key in `.env`:

```
GOOGLE_API_KEY=...
```

§1.2 below also compares against **docling** and **markitdown**. They aren't in
`requirements.txt` because docling alone pulls in ~1.8GB (torch, transformers, an OCR
engine) and downloads model weights from Hugging Face Hub on first use — not something
to impose on everyone just to run the pipeline. Install them only if you want to
reproduce that part:

```bash
pip install -r requirements-parser-eval.txt
```

If they aren't installed, §1.2 degrades gracefully and just report what happened.

In [ ]:
from __future__ import annotations

import json, os, re, time
from pathlib import Path

import pdfplumber
import pymupdf
from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(Path(".env"))  # explicit path; find_dotenv() stack-walks and is fragile

PDF_PATH = Path("data/fy2024_analysis_of_revenue_and_expenditure.pdf")
assert PDF_PATH.exists(), f"Document not found at {PDF_PATH}"

MODEL = "gemini-3.1-flash-lite"

_key = os.getenv("GOOGLE_API_KEY", "")
print("PDF       :", PDF_PATH)
print("Model     :", MODEL)
print("API key   :", "loaded" if _key and _key != "your_key_here" else "MISSING - add it to .env")

PDF       : data/fy2024_analysis_of_revenue_and_expenditure.pdf
Model     : gemini-3.1-flash-lite
API key   : loaded


## 1. Parsing

### 1.1 Establishing the page-reference mapping

First, check that the task's page numbers actually line up with what the PDF reader sees. Some documents with cover page misaligns the internal page index and the page number, which would make us pull numbers from the wrong page.

In [ ]:
# Python/PyMuPDF uses 0-based indexing, so doc[4] is PDF page 5
doc = pymupdf.open(PDF_PATH)
print(f"PDF page count: {doc.page_count}\n")

# sampling pages 2 and 8 to see the format of the document extracted
for idx in (1, 7):
    lines = [l.strip() for l in doc[idx].get_text().splitlines() if l.strip()]
    print(f"PDF index {idx} (printed page {idx + 1}) - first 6 lines in extraction order:")
    for i, l in enumerate(lines[:6]):
        print(f"  [{i}] {l!r}")
    print()

PDF page count: 37

PDF index 1 (printed page 2) - first 6 lines in extraction order:
  [0] 'MINISTRY OF FINANCE'
  [1] '2'
  [2] 'EXPLANATORY NOTES'
  [3] 'This document summarises and provides relevant'
  [4] 'highlights of the FY2024 Revenue and Expenditure'
  [5] 'Estimates presented to Parliament on 16 February 2024.'

PDF index 7 (printed page 8) - first 6 lines in extraction order:
  [0] 'MINISTRY OF FINANCE'
  [1] '8'
  [2] 'Table 1.1'
  [3] 'Fiscal Position in FY2022 and FY2023'
  [4] 'BLANK'
  [5] 'Revised FY2023'



**Observation**: By eye, the page number belongs to a footer at the bottom of the pages, however upon extraction using PyMuPDF, the page number is actually the second line on every page right after the header `MINISTRY OF FINANCE`. This is because `get_text()` does not return lines in top-to-bottom visual order. It returns them in the PDF's internal content-stream order, i.e. the order the text objects were drawn.

Note that this conclusion is specific to using PyMuPDF


In [ ]:

def printed_page_number(page) -> int | None:
    """Read the page number the document prints on itself.

    It's a footer, but get_text() emits it in the first few lines so 
    lines[:3] is enough to find it.
    """
    lines = [l.strip() for l in page.get_text().splitlines() if l.strip()]
    for line in lines[:3]:
        if re.fullmatch(r"\d{1,3}", line):
            return int(line)
    return None

print(f"{'PDF index':>10} | {'printed':>7} | first heading")
print("-" * 64)
for idx in (4, 5, 7, 15, 19, 35):
    lines = [l.strip() for l in doc[idx].get_text().splitlines() if l.strip()]
    heading = next((l for l in lines[:5] if not re.fullmatch(r"\d{1,3}", l)
                    and l != "MINISTRY OF FINANCE"), "")
    print(f"{idx:>10} | {printed_page_number(doc[idx]):>7} | {heading[:44]}")

 PDF index | printed | first heading
----------------------------------------------------------------
         4 |       5 | 01 Update on Financial Year 2023
         5 |       6 | Tax collections are revised to $17.5 billion
         7 |       8 | Table 1.1
        15 |      16 | Table 2.1
        19 |      20 | Table 2.4
        35 |      36 | Glossary of Terms


**Result:** `printed page N == PDF index N-1`, exactly. The task's page references are usable as-is, and every cited page contains what the task says it does.

Throughout this notebook, `PAGES[n]` is keyed by the **printed** page number so the code reads the same way the task does.

### 1.2 Understanding the various parsing libraries

1. **MarkItDown**: Converts a document straight to Markdown. Note that MarkItDown's PDF converter uses pdfplumber as dependencies so it's not really an alternative to pdfplumber at the same abstraction level.

2. **Docling**: Runs an actual layout and table-structure analysis model over the page, rather than just reading the text layer. 

3. **PyMuPDF**: Fast, general-purpose PDF library for extracting text, images, blocks, and coordinates, as well as rendering and manipulating PDF pages. Good when you need low-level control and speed.

4. **PDFPlumber**: Focuses on precise PDF text and table extraction, with access to words, characters, lines, bounding boxes, and table structures. Good when you need to customise or extract structured data from specific regions of a PDF.

Since this document's numbers mostly live in tables, the fairest test is a field that's a table
lookup start to finish: **Field 3: total top-ups, Table 2.4 on page 20**. Page 8 (Table 1.1,
Field 5) runs alongside it as a harder case — its column headers are nested across several
rows, where page 20's are a single line.

Each library runs on those two pages below, each cut out to its own single-page PDF so no
library gets credit or blame for anything happening elsewhere in the document.

In [38]:
import tempfile

def isolate_page(pdf_path: Path, printed_page: int) -> Path:
    """Cut one printed page out to its own single-page PDF, for a fair per-page test."""
    src = pymupdf.open(pdf_path)
    single = pymupdf.open()
    single.insert_pdf(src, from_page=printed_page - 1, to_page=printed_page - 1)
    out_path = Path(tempfile.gettempdir()) / f"htx_page{printed_page}.pdf"
    single.save(out_path)
    return out_path

PAGE20_PDF = isolate_page(PDF_PATH, 20)
PAGE8_PDF  = isolate_page(PDF_PATH, 8)
print("Isolated test pages:", PAGE20_PDF.name, "and", PAGE8_PDF.name)

Isolated test pages: htx_page20.pdf and htx_page8.pdf


In [39]:
MARKITDOWN_OK = DOCLING_OK = False
try:
    from markitdown import MarkItDown
    MARKITDOWN_OK = True
except ImportError as e:
    print(f"MarkItDown unavailable: {e}")

try:
    from docling.document_converter import DocumentConverter
    docling_converter = DocumentConverter()   # reused below, so its model loads only once
    DOCLING_OK = True
except ImportError as e:
    print(f"Docling unavailable: {e}  (pip install -r requirements-parser-eval.txt)")

docling_docs: dict[int, object] = {}          # page number -> docling document, reused further down


def _best_of(fn, repeats: int = 5) -> tuple[str, float]:
    """Fastest of N runs. These parsers take single-digit ms, where one sample is
    mostly scheduler noise - min() is the usual choice for microbenchmarks."""
    best, text = float("inf"), None
    for _ in range(repeats):
        t0 = time.perf_counter()
        text = fn()
        best = min(best, time.perf_counter() - t0)
    return text, best


def _read_pdfplumber(pdf_path: Path) -> str:
    with pdfplumber.open(pdf_path) as pdf:
        return pdf.pages[0].extract_text()


def parse_all(pdf_path: Path, page_no: int) -> dict[str, tuple[str, float]]:
    """Run every parser over one single-page PDF.  -> {name: (text, seconds)}"""
    out = {
        "PyMuPDF":    _best_of(lambda: pymupdf.open(pdf_path)[0].get_text()),
        "pdfplumber": _best_of(lambda: _read_pdfplumber(pdf_path)),
    }

    if MARKITDOWN_OK:
        out["MarkItDown"] = _best_of(lambda: MarkItDown().convert(str(pdf_path)).text_content)

    if DOCLING_OK:
        # Single run - it costs seconds, and repeating it would dominate the notebook.
        t0 = time.perf_counter()
        doc = docling_converter.convert(str(pdf_path)).document
        secs = time.perf_counter() - t0
        docling_docs[page_no] = doc
        out["Docling"] = (doc.export_to_markdown(), secs)

    return out


def show(text: str, marker: str, extra_lines: int = 2, max_chars: int = 900) -> str:
    """From the START of the extracted text through the row we care about.

    The column headers sit at the top of the page and are what make any figure
    interpretable at all, so the window always starts at the beginning rather
    than centring on the marker row.
    """
    text = text.strip()
    lines = text.splitlines()
    cut = next((n for n, l in enumerate(lines) if marker in l), len(lines) - 1)
    window = "\n".join(lines[:cut + 1 + extra_lines])
    if len(window) > max_chars:
        window = window[:max_chars]
    return window if window == text else window + "\n..."


UNDER_TEST = [
    (PAGE20_PDF, 20, "Total",                "Table 2.4 - top-ups (Field 3)"),
    (PAGE8_PDF,   8, "Corporate Income Tax", "Table 1.1 - fiscal position (Field 5)"),
]

timings: dict[int, dict[str, float]] = {}

for pdf_path, page_no, marker, label in UNDER_TEST:
    print("=" * 78)
    print(f"PAGE {page_no}  -  {label}")
    print("=" * 78)
    results = parse_all(pdf_path, page_no)
    timings[page_no] = {name: secs for name, (_, secs) in results.items()}
    for name, (text, secs) in results.items():
        print(f"\n--- {name}  ({secs*1000:,.0f}ms) " + "-" * (50 - len(name)))
        print(show(text, marker))
    print()

if DOCLING_OK:
    print("=" * 78)
    print("Cost of Docling's structure, relative to each flat-text parser")
    print("=" * 78)
    print(f"{'':<16}" + "".join(f"{f'page {p}':>12}" for p in timings))
    for name in ("PyMuPDF", "pdfplumber", "MarkItDown"):
        cells_ = "".join(
            f"{timings[p]['Docling'] / timings[p][name]:>11,.0f}x" if name in timings[p]
            else f"{'-':>12}" for p in timings
        )
        print(f"  vs {name:<13}{cells_}")
    print("\nFlat-text parsers: best of 5 runs. Docling: a single run (page 20 carries its")
    print("one-off model load; page 8 is warm - and still slower, so the cost tracks table")
    print("size more than startup).")


PAGE 20  -  Table 2.4 - top-ups (Field 3)


[INFO] 2026-09-13 10:33:11,111 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-13 10:33:11,118 [RapidOCR] download_file.py:60: File exists and is valid: /Users/jinjianzuo/Documents/htx-take-home/.venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-13 10:33:11,118 [RapidOCR] main.py:63: Using /Users/jinjianzuo/Documents/htx-take-home/.venv/lib/python3.14/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-13 10:33:11,137 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-13 10:33:11,139 [RapidOCR] download_file.py:60: File exists and is valid: /Users/jinjianzuo/Documents/htx-take-home/.venv/lib/python3.14/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-13 10:33:11,139 [RapidOCR] main.py:63: Using /Users/jinjianzuo/Documents/htx-take-home/.venv/lib/python3.14/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-13 10:33:11,154 [RapidO


--- PyMuPDF  (2ms) -------------------------------------------
MINISTRY OF FINANCE 
 
20 
 
Table 2.4 
Top-ups to Endowment and Trust Funds in FY2024 
 
 
Estimated FY2024 
($ million) 
Goods and Services Tax Voucher Fund 
6,000 
Future Energy Fund 
5,000 
Edusave Endowment Fund  
2,000 
Financial Sector Development Fund 
2,000 
National Productivity Fund 
2,000 
National Research Fund 
1,800 
Progressive Wage Credit Scheme Fund 
1,000 
Skills Development Fund 
500 
Public Transport Fund 
50 
Legal Aid Fund 
2 
Total 
20,352

--- pdfplumber  (16ms) ----------------------------------------
Top-ups to Endowment and Trust Funds in FY2024 Table 2.4
Estimated FY2024
($ million)
Goods and Services Tax Voucher Fund 6,000
Future Energy Fund 5,000
Edusave Endowment Fund 2,000
Financial Sector Development Fund 2,000
National Productivity Fund 2,000
National Research Fund 1,800
Progressive Wage Credit Scheme Fund 1,000
Skills Development Fund 500
Public Transport Fund 50
Legal Aid Fund 2
Total 2

**PyMuPDF** splits each cell onto its own line

**PDFPlumber** keeps the row intact and self-describing. 

Neither knows that `20,352` is specifically the *Total* row — that binding still has to come from an LLM reading the surrounding words.

**MarkItDown breaks on both of this document's target tables.** Page 20, above, comes
back with every fund *name* first and every *value* after, in two separate runs — `Future
Energy Fund` and `5,000` end up nowhere near each other. Page 8 fails differently: it renders as a confidently-formatted Markdown pipe
table with `BLANK` — a literal spacer string in the PDF's text layer — promoted to a
column heading, and the real headings split across columns that don't line up with the
figures beneath. Which of the two you get depends on a per-page heuristic in its PDF
converter, and on which *other* pages are in the batch, so page-level output isn't
stable. **Rejected**

**Docling is the only one that returns real structure.** Both pages come back as Markdown
tables with headings attached to the columns they actually belong to — including page 8,
where `Corporate Income Tax | 23.07 | 24.26 | 28.38 | 23.0 | 17.0` lands under the right
headers instead of as five loose numbers. The ratio table above is the catch. Warm, on page 8, Docling costs **~57x pdfplumber**
and **~900x PyMuPDF**; page 20's larger ratios include its one-off model load. That is
before the ~1.8GB of weights behind it.
It isn't flawless either — `BLANK` still leaks into its page-8 header row. The difference
is that its data rows stay bound to the right columns, which is the part that decides
whether a figure can be read back correctly.

Both pdfplumber and PyMuPDF also ship a dedicated table-*detection* API —
`extract_tables()` and `find_tables()` — sitting between plain text and something like
Docling. Worth checking what those produce on the same two pages, since Table 2.4 has no
merged cells and nothing a person would find ambiguous.


In [40]:
def clip(row, width: int = 16) -> list:
    """Shorten long cells so wide tables stay readable in the output."""
    return [(c[:width] + "..") if isinstance(c, str) and len(c) > width else c for c in row]


def preview(rows: list, n: int = 5) -> list:
    """First n rows plus the last, without repeating it when the table is short."""
    return rows if len(rows) <= n + 1 else rows[:n] + [["..."]] + [rows[-1]]


for pdf_path, page_no in [(PAGE20_PDF, 20), (PAGE8_PDF, 8)]:
    with pdfplumber.open(pdf_path) as pdf:
        pl_tables = pdf.pages[0].extract_tables()
    mu_tables = pymupdf.open(pdf_path)[0].find_tables().tables

    print("=" * 78)
    print(f"PAGE {page_no}   pdfplumber extract_tables(): {len(pl_tables)} table(s)   "
          f"|   PyMuPDF find_tables(): {len(mu_tables)} table(s)")
    print("=" * 78)

    if pl_tables:
        print("\npdfplumber's grid:")
        for row in preview(pl_tables[0]):
            print("   ", clip(row))

    if mu_tables:
        print("\nPyMuPDF's grid:")
        for row in preview(mu_tables[0].extract()):
            print("   ", clip(row))
    print()


PAGE 20   pdfplumber extract_tables(): 1 table(s)   |   PyMuPDF find_tables(): 1 table(s)

pdfplumber's grid:
    ['', None, None, '', 'Estimated FY2024', '']
    [None, None, None, None, '($ million)', None]
    ['Goods and Servic..', None, None, '6,000', None, None]
    ['Future Energy Fu..', None, None, '5,000', None, None]
    ['Edusave Endowmen..', None, None, '2,000', None, None]
    ['...']
    ['', 'Total', '', '', '20,352', '']

PyMuPDF's grid:
    ['', None, None, '', 'Estimated FY2024', '']
    [None, None, None, None, '($ million)', None]
    ['Goods and Servic..', None, None, '6,000', None, None]
    ['Future Energy Fu..', None, None, '5,000', None, None]
    ['Edusave Endowmen..', None, None, '2,000', None, None]
    ['...']
    ['', 'Total', '', '', '20,352', '']

PAGE 8   pdfplumber extract_tables(): 0 table(s)   |   PyMuPDF find_tables(): 0 table(s)



On page 20 both find one table, and the two grids are **identical** — which isn't a
coincidence. PyMuPDF's `table.py` states outright that *"portions of this code have been
ported from pdfplumber"*, so these aren't two independent opinions; they're one algorithm
behind two front ends. Running both is a consistency check, not a hedge.

And "found a table" isn't "got it right". The `Total` row comes back as
`['', 'Total', '', '', '20,352', '']` — label in column 1, value in column 4, in a
6-column grid for what is visually a 2-column table, with every row above it `None`-padded
to match.

On page 8 — the harder table, and the one Field 5 needs — both find **nothing at all**.
MOF rules none of its tables with visible lines, and these detectors lean on ruled borders;
page 20 has just enough whitespace regularity to fake a grid, page 8 doesn't.

So the tier between flat text and a layout model gives you either a misaligned grid or no
grid. Neither is something to build the extraction on.


### 1.4 Decision and justification

**Chosen: `pdfplumber.extract_text()`** — provisionally. No LLM has run yet, so this is a
choice made on parsing behaviour and operational cost alone. Whether it is good *enough*
is an open question that §2 and §3 answer.

| Library | Row/column binding | Speed (1 page, warm) | Dependency footprint | Verdict |
|---|---|---|---|---|
| PyMuPDF | cell-per-line, ambiguous | fastest | trivial | fastest, but gives the LLM the least to work with — kept for the page-number check (§1.1) |
| pdfplumber | row-preserved, self-describing | ~2-10x PyMuPDF | trivial | **chosen** — the middle ground |
| MarkItDown | unstable; strategy depends on which *other* pages are in the batch | fast | trivial (wraps pdfplumber) | **rejected** — wrong in a way that looks right (§1.2) |
| Docling | genuinely structured (a real table object) | ~57x pdfplumber, ~900x PyMuPDF | ~1.8GB incl. torch/transformers; HF Hub download on first use | best output of the four (§1.2); **rejected** on latency and operational size |

Reasoning:

1. **pdfplumber sits between the fastest option and the best one.** PyMuPDF is quicker but
   splits every table cell onto its own line, so nothing in the text binds `28.38` to
   *Corporate Income Tax*. Docling resolves that binding properly but costs ~57x more per
   page. pdfplumber keeps the row intact at a fraction of Docling's cost, which makes it
   the reasonable first thing to try.

2. **Docling is rejected on latency and operational size, not on output.** It produced the
   best extraction of the four and that should be said plainly. What it costs is ~1.8GB of
   torch/transformers/OCR weights, a Hugging Face Hub download on first use, and seconds
   per page against milliseconds. That is a lot of machinery to commit to before knowing
   whether the cheap option is already sufficient.

3. **The table-detection middle ground is a dead end here.** `extract_tables()` and
   `find_tables()` return a misaligned grid on page 20 and nothing at all on page 8 (§1.2).
   MOF rules none of its tables with visible lines, and both detectors — the same
   algorithm, as it turns out — depend on ruled borders.

**What this defers, deliberately.** pdfplumber leaves a real gap: a row like
`Corporate Income Tax 28.38 28.03 (0.35) (1.2)` still needs the reader to work out which
number belongs to which column, from headers printed further up the page. Docling would
have removed that problem at the parsing layer. The bet here is that an LLM can close it
at the prompting layer instead — for far less than 1.8GB.

That is a testable claim, not an assumption. If those checks fail in ways traceable to
the parse rather than the prompt, this decision is the thing to revisit.


In [ ]:
# The whole document, parsed once with the chosen parser, keyed by PRINTED page number
# so the code reads the same way the task's page references do.
t0 = time.time()
with pdfplumber.open(PDF_PATH) as pdf:
    PAGES: dict[int, str] = {i + 1: (page.extract_text() or "") for i, page in enumerate(pdf.pages)}
print(f"Parsed {len(PAGES)} pages with pdfplumber in {time.time() - t0:.2f}s "
      f"({sum(map(len, PAGES.values())):,} chars)\n")


def context_for(page_numbers: list[int]) -> str:
    """Render selected pages as tagged blocks so the model can cite page numbers."""
    return "\n\n".join(
        f'<page number="{n}">\n{PAGES[n].strip()}\n</page>' for n in sorted(page_numbers)
    )

print(context_for([20])[:520], "...")


Parsed 37 pages with pdfplumber in 1.48s (56,475 chars)

<page number="20">
Top-ups to Endowment and Trust Funds in FY2024 Table 2.4
Estimated FY2024
($ million)
Goods and Services Tax Voucher Fund 6,000
Future Energy Fund 5,000
Edusave Endowment Fund 2,000
Financial Sector Development Fund 2,000
National Productivity Fund 2,000
National Research Fund 1,800
Progressive Wage Credit Scheme Fund 1,000
Skills Development Fund 500
Public Transport Fund 50
Legal Aid Fund 2
Total 20,352
MINISTRY OF FINANCE 20
</page> ...


## 2. Extraction

**Assumptions:**
1. The page number stated in the assessment will be given to the LLM. This will then allow scoping of the document to only the page(s) of interest.
2. The expected data type stated in the assessment will be given to the LLM. This will inform the LLM what data type to look out for.
3. The table or section name is given as well, not just the page number. Field descriptions name "Table 2.4", "section 1.2 Operating Revenue" and the "OVERALL FISCAL POSITION" row, because page-scoping alone proved insufficient — §2.2 shows a field taking a correctly-labelled figure off the wrong page.
4. Row structure found by reading the PDF is encoded into the field descriptions, e.g. *"that row contains 3 numbers: Actual FY2022, Estimated FY2023, Revised FY2023 — return the FIRST"*. This goes beyond what the assessment states; it comes from inspecting the document, and it is the prompting-layer substitute for the column structure pdfplumber does not preserve (§1.4).
5. Every extracted figure must carry a verbatim quote and the page it came from. The assessment does not ask for this. It is added so §3 can check provenance mechanically instead of taking the value on trust, and because a model that has to quote its source has less room to invent one.
6. The LLM never calculates. Each field is a value stated somewhere in the document, so where the assessment asks for a "YOY percentage difference" the document's own stated percentage is used rather than one derived from two figures (see §5.3).

### 2.1 Scope the context instead of stuffing the whole document

The full document is ~56k characters (~15k tokens) and would fit in a single Gemini prompt. It is still the wrong thing to send, because the target figures have **near-duplicate distractors**:

In [ ]:
print("Every line mentioning 'Corporate Income Tax', across the document:\n")
for n, text in PAGES.items():
    for line in text.splitlines():
        if "Corporate Income Tax" in line:
            print(f"  page {n:>2}: {line.strip()[:88]}")

Every line mentioning 'Corporate Income Tax', across the document:

  page  5: collections from Corporate Income Tax, Other Taxes, Vehicle Quota Premiums,
  page  5: Corporate Income Tax collections are revised to $28.4 billion, which is
  page  8: Corporate Income Tax 23.07 24.26 28.38 23.0 17.0
  page  9: Carbon Taxes, 3.3% Corporate Income Tax
  page 16: Corporate Income Tax 28.38 28.03 (0.35) (1.2)
  page 26: Corporate Income Tax 16,032 16,732 16,112 18,196 23,072 28,380 28,029
  page 27: Corporate Income Tax 3.1% 3.3% 3.3% 3.0% 3.4% 4.1% 3.9%
  page 37: main components are Corporate Income Tax, Primary Budget Position


Five different Corporate Income Tax values appear across the document (23.07, 24.26, 28.38, 28.03, …), each correct for a different fiscal year and column. Handing the model all of them and hoping it picks the right one converts a retrieval problem into a coin flip.

Scoping the context to the pages the task cites makes the task unambiguous, and cuts tokens ~85% which would save cost in a real production grade system. This is retrieval done with the page references the task supplies — a real system would substitute a vector search here, but the principle is the same: **narrow the context before asking.**

### 2.3 Schema — one per field

One schema per requested field, each call scoped to the page(s) that field comes from.
Scoping is what does the disambiguating, so the descriptions stay close to the task's own
wording rather than carrying navigation instructions.

That split matters: a single pooled call over pages 5, 6, 8 and 20 returned **23.0** for
the YOY field on 3 runs out of 3, taken from Table 1.1 on page 8, where the correct 17.0%
is in the page 5 prose. The description said to use page 5 and the model overrode it. A
page named in an instruction is a suggestion; a page absent from the context is a
guarantee.

Two shapes cover the task's types — a number with a unit, and a list of strings. Both carry
evidence (source page, verbatim quote) so §3 can check provenance for every field rather
than trusting the value.


In [88]:
class NumericFact(BaseModel):
    """One extracted figure, with the evidence that supports it.

    Evidence fields come BEFORE value/unit deliberately: structured output is generated
    field-by-field in schema order, each field conditioned only on the ones already
    written. Quoting the source text first, then reading value/unit off that quote,
    grounds the number in the quote instead of letting the two be produced independently
    and merely checked against each other after the fact.
    """

    source_page: int = Field(description="The printed page number this value was read from.")
    quote: str = Field(description=(
        "The verbatim sentence or table row containing this value. Copy it exactly."
    ))
    value: float = Field(description=(
        "The numeric value as a plain number, read from the quote above. Strip currency "
        "symbols, '%' signs and thousands separators. A figure in parentheses is NEGATIVE: "
        "'(0.35)' -> -0.35."
    ))
    unit: str = Field(description=(
        "The unit exactly as the document expresses it, e.g. '$ billion', '$ million', 'percent'."
    ))


class TextListFact(BaseModel):
    """An extracted list of names rather than a figure - same evidence requirement."""
    source_pages: list[int] = Field(description="The printed page number(s) these were read from.")
    quote: str = Field(description="A verbatim sentence from the document naming at least one of them.")
    values: list[str] = Field(description="The list of names, each exactly as the document writes it.")


# --- one schema per field ----------------------------------------------------

class Field1CorporateIncomeTax(BaseModel):
    corporate_income_tax_2024: NumericFact = Field(
        description="Amount of Corporate Income Tax in 2024.")


class Field2CorporateIncomeTaxYoY(BaseModel):
    yoy_pct_difference_corp_income_tax_2024: NumericFact = Field(
        description=("YOY percentage difference of Corporate Income Tax in 2024 - the "
                     "percentage stated alongside that figure, not one you calculate."))


class Field3TotalTopUps(BaseModel):
    total_top_ups_2024: NumericFact = Field(
        description="Total amount of top ups in 2024.")


class Field4OperatingRevenueTaxes(BaseModel):
    taxes_in_operating_revenue: TextListFact = Field(
        description=("List of taxes mentioned in the section 'Operating Revenue'. Taxes "
                     "only - exclude non-tax revenue items such as Vehicle Quota Premiums, "
                     "Fees and Charges, and Statutory Boards' Contributions."))


class Field5LatestActualFiscalPosition(BaseModel):
    latest_actual_fiscal_position_billions: NumericFact = Field(
        description=("Latest Actual Fiscal Position in billions. 'Actual' is the operative "
                     "word: take the column of actual outturn, not one labelled Estimated "
                     "or Revised, which are forecasts. See assumption in section 5."))


print("Schemas:", [m.__name__ for m in (
    Field1CorporateIncomeTax, Field2CorporateIncomeTaxYoY, Field3TotalTopUps,
    Field4OperatingRevenueTaxes, Field5LatestActualFiscalPosition)])


Schemas: ['Field1CorporateIncomeTax', 'Field2CorporateIncomeTaxYoY', 'Field3TotalTopUps', 'Field4OperatingRevenueTaxes', 'Field5LatestActualFiscalPosition']


### 2.4 Prompt v1

The system prompt carries the *behavioural* rules — grounding, sign conventions, fiscal-year discipline. The schema carries the *per-field* instructions. Keeping them separate avoids restating five field definitions in prose, and lets the same system prompt serve all three calls.

Rule 5 is the guardrail against the distractor problem from §2.1–2.2.

In [89]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

SYSTEM_V1 = """You are a meticulous financial-document analyst working with Singapore Government Budget documents.

Rules:
1. Extract ONLY what the supplied pages state. Never infer, calculate, or recall a figure from memory. The pages below are the sole source of truth.
2. Each page is wrapped in <page number="N"> tags. N is the document's printed page number - use it for source_page.
3. Quotes must be copied verbatim from the page text.
4. Figures in parentheses are negative: (3.57) means -3.57.
5. Fiscal years matter. This document reports both FY2023 (Actual, Estimated and Revised) and FY2024 (Estimated), and the same line item appears with a DIFFERENT value for each. Identify which one the field asks for and take only that one. If a field description names a specific page, section or row, use that source and no other.
6. If a requested value is genuinely absent from the supplied pages, say so rather than guessing."""

USER = """Extract the requested fields from these pages of the FY2024 Analysis of Revenue and Expenditure (Singapore Ministry of Finance).

{context}"""

llm = ChatGoogleGenerativeAI(model=MODEL)

def build(schema, system_prompt):
    """Prompt -> LLM with structured output. The same chain serves every prompt version."""
    prompt = ChatPromptTemplate.from_messages([("system", system_prompt), ("human", USER)])
    return prompt | (
        llm.with_structured_output(schema)
           .with_retry(
               stop_after_attempt=5,
               wait_exponential_jitter=True,
               exponential_jitter_params={"initial": 4, "max": 60, "exp_base": 2, "jitter": 2},
           )
    )

print("Prompt v1 and chain factory ready.")

Prompt v1 and chain factory ready.


### 2.5 Run — one call per field


In [90]:
# Pages each field comes from, in field order. Each call sees only these pages.
FIELD_PAGES = [[5], [5], [20], [5, 6], [8]]


def run_extraction(schemas: list, system_prompt: str) -> dict:
    """One LLM call per field. Used unchanged for every prompt version."""
    extracted = {}
    for schema, pages in zip(schemas, FIELD_PAGES):
        result = build(schema, system_prompt).invoke({"context": context_for(pages)})
        extracted.update(result.model_dump())
    return extracted


SCHEMAS_V1 = [Field1CorporateIncomeTax, Field2CorporateIncomeTaxYoY, Field3TotalTopUps,
              Field4OperatingRevenueTaxes, Field5LatestActualFiscalPosition]

extracted_v1 = run_extraction(SCHEMAS_V1, SYSTEM_V1)
print(f"{len(SCHEMAS_V1)} LLM calls complete.")

5 LLM calls complete.


In [91]:
def show(extracted: dict) -> None:
    """Print every extracted field with its evidence."""
    for name, field in extracted.items():
        print(name)
        for key, value in field.items():
            if isinstance(value, list) and value and isinstance(value[0], dict):
                print(f"    {key}:")
                for item in value:
                    print(f"        {item}")
            else:
                print(f"    {key}: {value}")
        print()


show(extracted_v1)

corporate_income_tax_2024
    source_page: 5
    quote: Corporate Income Tax collections are revised to $28.4 billion
    value: 28.4
    unit: $ billion

yoy_pct_difference_corp_income_tax_2024
    source_page: 5
    quote: Corporate Income Tax collections are revised to $28.4 billion, which is $4.1 billion (17.0%) higher than the Estimated FY2023 figure due to stronger-than-expected economic growth in 2022.
    value: 17.0
    unit: percent

total_top_ups_2024
    source_page: 20
    quote: Total 20,352
    value: 20352.0
    unit: $ million

taxes_in_operating_revenue
    source_pages: [5, 6]
    quote: This increase is mainly due to higher collections from Corporate Income Tax, Other Taxes, Vehicle Quota Premiums, Personal Income Tax, Assets Taxes, and Betting Taxes, partially offset by lower collections from the Goods and Services Tax.
    values: ['Corporate Income Tax', 'Other Taxes', 'Vehicle Quota Premiums', 'Personal Income Tax', 'Assets Taxes', 'Betting Taxes', 'Goods and Se

## 3. Validation

The ground-truth values below were read by hand from the PDF. Without this, "the LLM returned a number" is indistinguishable from "the LLM returned the right number" — and as §2.2 shows, the difference is real.

Three classes of check, folded into one verdict per field:
- **Value** — does the number match the hand-read truth? Where the cited page does not contain the requested fiscal year, the truth is **null**, and any number is a failure: it means the model substituted a different year to fill the slot.
- **Coverage** — does the tax list contain every tax named in §1.2, and **nothing that isn't a tax**? Vehicle Quota Premiums sits in the same sentence as the taxes but is a separate revenue line.
- **Provenance** — does each quoted sentence genuinely appear on the page the model cited? This catches a fabricated citation even when the value happens to be right.

In [92]:
GROUND_TRUTH = {
    # Page 5 is chapter "01 Update on Financial Year 2023": its $28.4bn and 17.0% are Revised
    # FY2023 figures. The cited page has no 2024 figure, so the correct answer is null.
    "corporate_income_tax_2024":               None,
    "yoy_pct_difference_corp_income_tax_2024": None,
    "total_top_ups_2024":                      20352.0,   # p20: Table 2.4, Total row, $ million
    "latest_actual_fiscal_position_billions":  1.72,      # p8:  Table 1.1, OVERALL FISCAL POSITION, Actual FY2022 column
}

REQUIRED_TAXES = {
    "Corporate Income Tax", "Personal Income Tax", "Assets Taxes", "Betting Taxes",
    "Goods and Services Tax", "Casino Taxes",
    "Foreign Worker Levy", "Water Conservation Tax", "Land Betterment Charge", "Annual Tonnage Tax",
}
# Named in the section and defined by the document as taxes; fine either way.
# Anything else in the list (e.g. Vehicle Quota Premiums) fails the field.
OPTIONAL_TAXES = {"Other Taxes"}


def taxes_from(field: dict) -> list[str]:
    """Prompt v1 returns the tax names directly. Prompt v2 returns every revenue item with an is_tax flag."""
    if "values" in field:
        return field["values"]
    return [item["name"] for item in field["items"] if item["is_tax"]]


def quote_on_page(quote: str | None, pages: list[int]) -> bool:
    """True if the first 60 characters of the quote appear on the cited page(s), ignoring line breaks."""
    if not quote:
        return False
    page_text = " ".join(" ".join(PAGES.get(p, "").split()) for p in pages)
    return " ".join(quote.split())[:60] in page_text


def check_number(field: str, extracted: dict) -> tuple[bool, str]:
    got, want = extracted[field]["value"], GROUND_TRUTH[field]
    if want is None:
        return (True, "null, as expected") if got is None else (False, f"expected null, got {got}")
    if got is None:
        return False, f"expected {want}, got null"
    if abs(got - want) > 0.01 * abs(want):
        return False, f"expected {want}, got {got}"
    if not quote_on_page(extracted[field]["quote"], [extracted[field]["source_page"]]):
        return False, "right value, but the quote is not on the cited page"
    return True, f"{got}, as expected"


def check_taxes(extracted: dict) -> tuple[bool, str]:
    field = extracted["taxes_in_operating_revenue"]
    got = set(taxes_from(field))
    missing = REQUIRED_TAXES - got
    not_taxes = got - REQUIRED_TAXES - OPTIONAL_TAXES
    if missing or not_taxes:
        return False, f"missing {sorted(missing)}, not taxes {sorted(not_taxes)}"
    if not quote_on_page(field["quote"], field["source_pages"]):
        return False, "right list, but the quote is not on the cited pages"
    return True, "all taxes, nothing extra"


def score(extracted: dict) -> dict[str, tuple[bool, str]]:
    """(passed, reason) for each field."""
    results = {field: check_number(field, extracted) for field in GROUND_TRUTH}
    results["taxes_in_operating_revenue"] = check_taxes(extracted)
    return results


def validate(extracted: dict) -> None:
    for field, (ok, reason) in score(extracted).items():
        print(f"  [{'PASS' if ok else 'FAIL'}]  {field:<42} {reason}")


validate(extracted_v1)

  [FAIL]  corporate_income_tax_2024                  expected null, got 28.4
  [FAIL]  yoy_pct_difference_corp_income_tax_2024    expected null, got 17.0
  [PASS]  total_top_ups_2024                         20352.0, as expected
  [PASS]  latest_actual_fiscal_position_billions     1.72, as expected
  [FAIL]  taxes_in_operating_revenue                 missing ['Annual Tonnage Tax', 'Casino Taxes', 'Foreign Worker Levy', 'Land Betterment Charge', 'Water Conservation Tax'], not taxes ['Vehicle Quota Premiums']


### 3.1 What prompt v1 gets wrong

- **Fields 1 and 2 return $28.4bn and 17.0%** — the Revised FY2023 figures on page 5 — instead of null. The schema can't say "not here": `value` is a required `float`, so the model has to put *some* number in it, even though system rule 6 tells it to say when a value is absent.
- **Field 5 returns a forecast column** (Estimated or Revised FY2023) instead of 1.72, the Actual FY2022 column. Table 1.1's headers are split over several lines, and the schema goes straight from `quote` to `value`, so nothing makes the model match numbers to columns — and there's no way to see why it picked what it picked.
- **Field 4 copies one summary sentence as the list.** Vehicle Quota Premiums gets in, even though the field description excludes it, and the taxes named in later sentences (Foreign Worker Levy, Water Conservation Tax, Land Betterment Charge, Annual Tonnage Tax, Casino Taxes) are missed.

Field 3 is fine: one column, one `Total` row.

### 3.2 Prompt v2

Same pages, same `run_extraction()`, same validation — only the schemas and system prompt change, one change per problem above:

1. **Fields 1–2: allow null.** `value`, `unit`, `quote` and `source_page` can be null. Each field states the fiscal year it needs, and the system prompt says a figure for a different year is not an answer.
2. **Field 5: define the terms and show the work.** The system prompt defines Actual / Estimated / Revised and explains how split table headers read. The schema adds `reasoning`, `fiscal_year` and `basis` *before* `value`, so the model pairs numbers with columns in writing before it picks one — and we can read why.
3. **Field 4: classify, then filter in code.** The model lists every revenue item in the section with `is_tax` and a reason; `taxes_from()` keeps the taxes.

In [93]:
from typing import Literal

SYSTEM_V2 = """You are a meticulous financial-document analyst working with Singapore Government Budget documents.

Rules:
1. Extract ONLY what the supplied pages state. Never infer, calculate, or recall a figure from memory. The pages below are the sole source of truth.
2. Each page is wrapped in <page number="N"> tags. N is the document's printed page number - use it for source_page.
3. Quotes must be copied verbatim from the page text.
4. Figures in parentheses are negative: (3.57) means -3.57.
5. Every figure in this document belongs to one fiscal year AND one basis:
   - Actual: the realised outturn of a fiscal year that has closed.
   - Estimated: the forecast published at Budget time for that year.
   - Revised: an updated forecast made later, during the year. Still a forecast, not an actual.
   The same line item has a different value for each combination (e.g. Actual FY2022, Estimated FY2023, Revised FY2023).
6. Each field states the fiscal year and/or basis it requires. If no figure on the supplied pages matches, set value to null and say why in reasoning. Never substitute a different year or basis: a number from the wrong year is worse than no number.
7. Tables were extracted from a PDF as plain text. A column header may be split over several lines above the rows, and "BLANK" is a layout placeholder to ignore. Rebuild each header by reading those lines top to bottom, then pair a row's numbers with the columns left to right. When a row has fewer numbers than there are columns, the trailing columns (usually the % change ones) are the empty ones."""


class NumericFactV2(BaseModel):
    """Like NumericFact, but nullable, and with the reasoning written before the value."""

    source_page: int | None = Field(description="The printed page number the value was read from. Null if no value qualifies.")
    quote: str | None = Field(description="The verbatim sentence or table row the value comes from. Copy it exactly.")
    reasoning: str = Field(description=(
        "If the quote is a table row, name each column header and the number under it. Then say which "
        "fiscal year and basis the chosen number has, and why it matches what the field requires - "
        "or why nothing on the pages does."))
    fiscal_year: str | None = Field(description="Fiscal year of the value, e.g. 'FY2022'. Null if no value qualifies.")
    basis: Literal["Actual", "Estimated", "Revised"] | None = Field(description="Basis of the value. Null if no value qualifies.")
    value: float | None = Field(description=(
        "The number as a plain float, read from the quote. Strip currency symbols, '%' signs and thousands "
        "separators; parentheses mean negative. Null if nothing on the pages matches the required fiscal "
        "year and basis."))
    unit: str | None = Field(description="The unit as the document expresses it, e.g. '$ billion', '$ million', 'percent'.")


class RevenueItem(BaseModel):
    name: str = Field(description="The item's name exactly as the document writes it.")
    reason: str = Field(description="The words in the text that show whether it is a tax.")
    is_tax: bool = Field(description=(
        "True if the text names the item as part of a tax category ('X Taxes, which include A and B' "
        "makes A and B taxes), whatever its own name says. Otherwise true only if its name contains "
        "Tax, Taxes, Duty or Levy."))


class RevenueItemsFact(BaseModel):
    source_pages: list[int] = Field(description="The printed page number(s) the items were read from.")
    items: list[RevenueItem] = Field(description=(
        "EVERY revenue item named anywhere in the section, tax or not. Go sentence by sentence, including "
        "items named inside another item's description. Keep non-taxes in the list with is_tax false."))
    quote: str = Field(description="A verbatim sentence from the section naming at least one of the items.")


class Field1CorporateIncomeTaxV2(BaseModel):
    corporate_income_tax_2024: NumericFactV2 = Field(description=(
        "Amount of Corporate Income Tax in 2024. Required fiscal year: FY2024, any basis. "
        "A Corporate Income Tax figure for another fiscal year does not qualify."))


class Field2CorporateIncomeTaxYoYV2(BaseModel):
    yoy_pct_difference_corp_income_tax_2024: NumericFactV2 = Field(description=(
        "YOY percentage difference of Corporate Income Tax in 2024 - the percentage change the document "
        "states for FY2024, not one you calculate. Required fiscal year: FY2024. A percentage describing "
        "another fiscal year's figure does not qualify."))


class Field3TotalTopUpsV2(BaseModel):
    total_top_ups_2024: NumericFactV2 = Field(description=(
        "Total amount of top ups in 2024. Required fiscal year: FY2024, any basis."))


class Field4OperatingRevenueTaxesV2(BaseModel):
    taxes_in_operating_revenue: RevenueItemsFact = Field(description=(
        "Every revenue item mentioned in the section 'Operating Revenue', each classified as a tax or not."))


class Field5LatestActualFiscalPositionV2(BaseModel):
    latest_actual_fiscal_position_billions: NumericFactV2 = Field(description=(
        "Latest Actual Fiscal Position in billions. Required basis: Actual - of the columns labelled "
        "Actual, take the most recent fiscal year. A column labelled Estimated or Revised does not "
        "qualify, even for a later year."))


SCHEMAS_V2 = [Field1CorporateIncomeTaxV2, Field2CorporateIncomeTaxYoYV2, Field3TotalTopUpsV2,
              Field4OperatingRevenueTaxesV2, Field5LatestActualFiscalPositionV2]

In [94]:
extracted_v2 = run_extraction(SCHEMAS_V2, SYSTEM_V2)
show(extracted_v2)

corporate_income_tax_2024
    source_page: None
    quote: None
    reasoning: The provided page 5 contains data for 'Revised FY2023' and 'Estimated FY2023'. It does not contain any data for 'FY2024' (either Actual, Estimated, or Revised).
    fiscal_year: None
    basis: None
    value: None
    unit: None

yoy_pct_difference_corp_income_tax_2024
    source_page: None
    quote: None
    reasoning: The provided pages contain information specifically for FY2023 (Actual vs Revised vs Estimated). There is no data provided for FY2024 or year-over-year percentage comparisons for the 2024 fiscal year.
    fiscal_year: None
    basis: None
    value: None
    unit: None

total_top_ups_2024
    source_page: 20
    quote: Total 20,352
    reasoning: The table header indicates the values are 'Estimated FY2024'. The row 'Total' lists the value '20,352' under the column associated with top-ups, representing the total amount for FY2024 (Estimated).
    fiscal_year: FY2024
    basis: Estimated
    

In [ ]:
validate(extracted_v2)

v1_scores, v2_scores = score(extracted_v1), score(extracted_v2)
print(f"\n  {'field':<42}{'v1':>6}{'v2':>6}")
for field in v1_scores:
    v1 = "PASS" if v1_scores[field][0] else "FAIL"
    v2 = "PASS" if v2_scores[field][0] else "FAIL"
    print(f"  {field:<42}{v1:>6}{v2:>6}")

  [PASS]  corporate_income_tax_2024                  null, as expected
  [PASS]  yoy_pct_difference_corp_income_tax_2024    null, as expected
  [PASS]  total_top_ups_2024                         20352.0, as expected
  [PASS]  latest_actual_fiscal_position_billions     1.72, as expected
  [PASS]  taxes_in_operating_revenue                 all taxes, nothing extra

  field                                         v1    v2
  corporate_income_tax_2024                   FAIL  PASS
  yoy_pct_difference_corp_income_tax_2024     FAIL  PASS
  total_top_ups_2024                          PASS  PASS
  latest_actual_fiscal_position_billions      FAIL  PASS
  taxes_in_operating_revenue                  FAIL  PASS


## 4. An ambiguity in field 1 that changes the answer

The task asks for *"Amount of Corporate Income Tax in **2024**"* but cites **page 5**. Page 5 sits inside chapter *"01 Update on Financial Year 2023"*, so its figures ($28.4bn, +17.0%) are Revised FY2023, not FY2024. The FY2024 estimate is in Table 2.1 on page 16.

| Reading | Source | CIT | YoY |
|---|---|---|---|
| **A — stay on the cited page** | p5 has no FY2024 figure | **null** | **null** |
| **B — follow the words "in 2024"** | p16, Table 2.1, Estimated FY2024 | **$28.03bn** | **−1.2%** |

**Adopted: Reading A.** Every other page reference in the task is exact (top-ups p20 ✓, fiscal position p8 ✓, Operating Revenue section p5–6 ✓, estate duty p36 ✓), so the page reference governs — and page 5 has no 2024 figure. Returning page 5's $28.4bn would answer a different question (FY2023) with a confident-looking number, which is exactly what prompt v1 did (§3.1).


## 5. Assumptions

1. **Page references are 1-indexed printed page numbers**, verified in §1.1 to equal PDF index + 1.

2. **"Corporate Income Tax in 2024" = null.** The cited page 5 reports only Revised FY2023 figures ($28.4bn, +17.0%); it has no FY2024 figure. See §4 — the FY2024 estimate on page 16 is extracted as the alternative.

3. **"YOY percentage difference" = null, for the same reason.** Page 5's 17.0% compares Revised FY2023 with Estimated FY2023 — a revision within one fiscal year, not a 2024 figure. The LLM never calculates a percentage from two figures.

4. **"Total amount of top ups in 2024" = the Table 2.4 Total, `20352.0` in $ million.** The table is denominated in $ million, so that is the raw value, and the schema captures the unit explicitly rather than assuming it.

5. **"Latest Actual Fiscal Position" = `1.72`, the Actual FY2022 column of Table 1.1.** The table's three figure columns are Actual FY2022, Estimated FY2023 and Revised FY2023. Only the first is an actual; the other two are forecasts.

6. **The tax list includes only items the document names as taxes.** Section 1.2 also mentions Vehicle Quota Premiums, an Operating Revenue item that is not a tax. Items named as part of "Other Taxes" (Foreign Worker Levy, Water Conservation Tax, Land Betterment Charge, Annual Tonnage Tax) and Casino Taxes, mentioned under Betting Taxes, **are** included. "Other Taxes" itself is accepted either way.

7. **Each prompt version was run once, with the model's default settings.** The results in §3 are from a single run of prompt v1 and a single run of prompt v2.

### What I would add with more time

- **Repeat-run agreement.** Run each extraction several times and report agreement as a confidence signal; disagreement flags a field for human review.
- **A cheap self-check pass.** Feed each `(value, quote, page)` triple back with the source page and ask a second call to verify the quote supports the value — catching a wrong column or year automatically rather than via hand-written ground truth.
- **Generalising beyond fixed page hints.** The page numbers come from the task. A production version would locate sections by heading (`1.2 Operating Revenue`, `Table 2.4`) and fall back to embedding search, so the pipeline survives next year's edition with different pagination.